# 1. Exploratory analysis and linear baselines

Sixty CNC/CNF/rGO films from Tables 2 and 3 of Ramezani et al., *Micromachines* 16(4), 393 (2025), doi:10.3390/mi16040393: 30 reduced with citric acid and 30 with L-ascorbic acid. Targets are conductivity, tensile strength and thickness.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from ddmo.data import FEATURES, TARGETS, design_matrix, load_films, variance_inflation
summary = json.loads((ROOT / "results" / "summary.json").read_text())
df = load_films()
X, Y = design_matrix(df)
pd.set_option("display.precision", 3)
pd.set_option("display.notebook_repr_html", False)
pd.set_option("display.width", 120)

In [2]:
df.groupby("agent")[TARGETS].describe().T.round(2)

agent                       ascorbic  citric
conductivity_S_per_m count     30.00   30.00
                     mean       0.98    0.90
                     std        0.38    0.38
                     min        0.32    0.28
                     25%        0.70    0.60
                     50%        0.94    0.86
                     75%        1.25    1.23
                     max        1.72    1.62
tensile_strength_MPa count     30.00   30.00
                     mean      36.97   35.23
                     std        5.14    5.12
                     min       28.00   26.00
                     25%       33.00   31.25
                     50%       36.50   35.00
                     75%       41.00   39.75
                     max       46.00   44.00
thickness_um         count     30.00   30.00
                     mean      99.23  103.50
                     std       22.59   23.21
                     min       52.00   54.00
                     25%       85.50   90.50
          

Temperature is 95 °C for every film, so the design matrix uses the five varying inputs and an agent indicator.

In [3]:
corr = pd.DataFrame(np.corrcoef(np.column_stack([X, Y]), rowvar=False),
                    index=FEATURES + TARGETS, columns=FEATURES + TARGETS)
corr.round(2)

                      CNC_wt_pct  CNF_wt_pct  rGO_wt_pct  acid_M    pH  ascorbic  conductivity_S_per_m  \
CNC_wt_pct                  1.00       -0.27        0.92    0.63 -0.55     -0.02                  0.96   
CNF_wt_pct                 -0.27        1.00       -0.08    0.03  0.06     -0.02                 -0.11   
rGO_wt_pct                  0.92       -0.08        1.00    0.64 -0.53     -0.02                  0.98   
acid_M                      0.63        0.03        0.64    1.00 -0.95     -0.73                  0.55   
pH                         -0.55        0.06       -0.53   -0.95  1.00      0.83                 -0.44   
ascorbic                   -0.02       -0.02       -0.02   -0.73  0.83      1.00                  0.11   
conductivity_S_per_m        0.96       -0.11        0.98    0.55 -0.44      0.11                  1.00   
tensile_strength_MPa        0.94       -0.10        0.97    0.50 -0.37      0.17                  0.99   
thickness_um                0.86        0.09  

In [4]:
pd.Series(variance_inflation(df), name="VIF")

CNC_wt_pct    25.12
CNF_wt_pct     1.98
rGO_wt_pct     9.38
acid_M        18.96
pH            49.99
ascorbic      41.33
Name: VIF, dtype: float64

**Reading.** rGO loading correlates with conductivity at r = 0.98. Acid concentration, pH and the reducing agent are nearly interchangeable (VIF 19–50), and CNC and rGO move together (r = 0.92). Any per-feature attribution inside those groups is fragile, which shapes every later choice: regularization, stability analysis and grouped SHAP.

In [5]:
rows = []
for name in ["mean", "ols"]:
    s, g = summary["nested_cv"][name], summary["grouped_cv"][name]
    rows.append({"model": s["label"], "nested CV R2": s["r2_mean"], "+/-": s["r2_std"],
                 **{f"R2 {t}": v for t, v in s["r2_per_target"].items()},
                 "composition-grouped R2": g["r2_mean"]})
pd.DataFrame(rows).set_index("model")

                   nested CV R2    +/-  R2 conductivity_S_per_m  R2 tensile_strength_MPa  R2 thickness_um  \
model                                                                                                       
Mean predictor           -0.082  0.080                   -0.099                   -0.084           -0.064   
Linear regression         0.958  0.022                    0.996                    0.989            0.888   

                   composition-grouped R2  
model                                      
Mean predictor                     -0.053  
Linear regression                   0.965  

Linear regression already explains most of the variance out of sample. Thickness is the hardest target for a linear model, which the nonlinear model in notebook 3 addresses. The composition-grouped column holds out every film of a composition together, so it tests prediction for compositions never seen.